In [10]:
import os
import pandas as pd

In [11]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
SCRIPT_DIR_PATH = os.getcwd()
ROOT_DIR_PATH = os.path.dirname(SCRIPT_DIR_PATH)
DATA_DIR_PATH = os.path.join(ROOT_DIR_PATH, "data")
PROCESSED_DATA_DIR_PATH = os.path.join(DATA_DIR_PATH, "processed_data")
RESULTS_DIR_PATH = os.path.join(SCRIPT_DIR_PATH, "results")
OUTPUT_DIR_PATH = os.path.join(SCRIPT_DIR_PATH, "output")
ENSEMBLE_DIR_PATH = os.path.join(OUTPUT_DIR_PATH, "ensemble")
MODELS_DIR_PATH = os.path.join(OUTPUT_DIR_PATH, "models")
TRAINING_DIR_PATH = os.path.join(OUTPUT_DIR_PATH, "training")

# Make sure the output directories exist
os.makedirs(ENSEMBLE_DIR_PATH, exist_ok=True)
os.makedirs(MODELS_DIR_PATH, exist_ok=True)
os.makedirs(TRAINING_DIR_PATH, exist_ok=True)

## Load data

In [4]:
training_df_log_transformed = pd.read_csv(
    os.path.join(TRAINING_DIR_PATH, "training_df_log_transformed.csv"))

training_df_log_transformed.head()

,iso_alpha_3,income_group,region,year,pop_growth,renewable_energy_consumption_pct,forest_area_pct,gdp_growth_pct,gdp_per_capita_growth_pct,manufacturing_pct_of_gdp,log_pop_total,log_gdp_2021_ppp_intl_usd,log_gdp_per_capita_2021_ppp_intl_usd,log_imports_pct_of_gdp,log_industry_pct_of_gdp,log_exports_pct_of_gdp,log_total_emissions,log_gdp_2021_ppp_intl_usd_lag1,log_pop_total_lag1
0,ABW,High income,Latin America & Caribbean,2001,0.935033,0.2,2.333333,4.182002,3.212406,4.277073,11.423438,22.032621,10.609219,4.254113,2.642912,4.276621,0.295751,21.991652,11.414088
1,ABW,High income,Latin America & Caribbean,2002,0.692052,0.2,2.333333,-0.944953,-1.628099,4.425834,11.430359,22.023127,10.592804,4.243719,2.656169,4.183032,0.309851,22.032621,11.423438
2,ABW,High income,Latin America & Caribbean,2003,1.138229,0.2,2.333333,1.110505,-0.033839,4.777505,11.441741,22.034171,10.592466,4.263568,2.765829,4.154674,0.345181,22.023127,11.430359
3,ABW,High income,Latin America & Caribbean,2004,2.135358,0.2,2.333333,7.293728,5.026912,5.622020,11.463094,22.104571,10.641511,4.230700,2.809192,4.184575,0.357805,22.034171,11.441741
4,ABW,High income,Latin America & Caribbean,2005,2.590757,0.2,2.333333,-0.383138,-2.930823,6.187295,11.489002,22.100732,10.611765,4.356388,2.850560,4.235197,0.380248,22.104571,11.463094


## Create Projections

In [12]:
from utils.ml_utils_v2 import EnsembleProjections
ep = EnsembleProjections()

In [13]:
numeric_cols = training_df_log_transformed.select_dtypes(include=["float64", "int64"]).columns.tolist()
numeric_cols_to_drop = ["year", "log_total_emissions"]
numeric_cols = [col for col in numeric_cols if col not in numeric_cols_to_drop]
numeric_cols

['pop_growth',
 'renewable_energy_consumption_pct',
 'forest_area_pct',
 'gdp_growth_pct',
 'gdp_per_capita_growth_pct',
 'manufacturing_pct_of_gdp',
 'log_pop_total',
 'log_gdp_2021_ppp_intl_usd',
 'log_gdp_per_capita_2021_ppp_intl_usd',
 'log_imports_pct_of_gdp',
 'log_industry_pct_of_gdp',
 'log_exports_pct_of_gdp',
 'log_gdp_2021_ppp_intl_usd_lag1',
 'log_pop_total_lag1']

In [14]:
# Arima params:
n_scenarios = 10

In [8]:
ensemble_arima_df = ep.generate_ensemble_arima(
    df=training_df_log_transformed,
    feature_cols=numeric_cols,
    start_year=2022,
    end_year=2030,
    n_scenarios=n_scenarios,
    arima_order=(1,1,1),
    random_state=42
)


In [15]:
# If you don't want to generate auto-tuned ARIMA models, you can comment this out
ensemble_arima_tuned_df = ep.generate_ensemble_arima(
    df=training_df_log_transformed,
    feature_cols=numeric_cols,
    start_year=2022,
    end_year=2030,
    n_scenarios=n_scenarios,
    auto_tune_arima=True,
    random_state=42
)

tunned arima order:  (0, 1, 0)
tunned arima order:  (0, 1, 0)
tunned arima order:  (0, 0, 0)
tunned arima order:  (0, 0, 1)


/Users/javier/miniconda3/envs/etpe_env/lib/python3.12/site-packages/pmdarima/arima/auto.py:444: UserWarning: Input time-series is completely constant; returning a (0, 0, 0) ARMA.
  warnings.warn('Input time-series is completely constant; '


tunned arima order:  (0, 0, 1)
tunned arima order:  (1, 0, 0)
tunned arima order:  (0, 1, 2)
tunned arima order:  (0, 0, 0)
tunned arima order:  (1, 1, 2)
tunned arima order:  (1, 0, 0)
tunned arima order:  (1, 0, 0)
tunned arima order:  (0, 0, 1)
tunned arima order:  (0, 0, 0)
tunned arima order:  (1, 1, 1)
tunned arima order:  (0, 0, 0)
tunned arima order:  (0, 1, 0)
tunned arima order:  (0, 0, 0)


/Users/javier/miniconda3/envs/etpe_env/lib/python3.12/site-packages/pmdarima/arima/auto.py:444: UserWarning: Input time-series is completely constant; returning a (0, 0, 0) ARMA.
  warnings.warn('Input time-series is completely constant; '


tunned arima order:  (2, 1, 0)
tunned arima order:  (2, 1, 0)
tunned arima order:  (0, 1, 0)
tunned arima order:  (1, 1, 0)
tunned arima order:  (1, 1, 1)
tunned arima order:  (0, 1, 0)
tunned arima order:  (0, 0, 0)
tunned arima order:  (0, 1, 0)
tunned arima order:  (0, 0, 0)
tunned arima order:  (0, 1, 0)
tunned arima order:  (0, 1, 1)
tunned arima order:  (2, 0, 0)
tunned arima order:  (0, 1, 0)
tunned arima order:  (0, 1, 0)
tunned arima order:  (0, 1, 1)
tunned arima order:  (0, 1, 1)
tunned arima order:  (0, 1, 0)
tunned arima order:  (2, 1, 0)
tunned arima order:  (1, 1, 0)
tunned arima order:  (2, 0, 1)
tunned arima order:  (0, 1, 0)
tunned arima order:  (0, 1, 0)
tunned arima order:  (0, 1, 0)
tunned arima order:  (2, 1, 0)
tunned arima order:  (2, 1, 0)
tunned arima order:  (2, 0, 0)
tunned arima order:  (0, 1, 0)
tunned arima order:  (1, 1, 0)
tunned arima order:  (0, 0, 0)
tunned arima order:  (0, 0, 0)
tunned arima order:  (0, 1, 1)
tunned arima order:  (2, 1, 0)
tunned a

...

tunned arima order:  (0, 0, 2)
tunned arima order:  (0, 1, 0)
tunned arima order:  (0, 1, 0)
tunned arima order:  (0, 0, 0)


/Users/javier/miniconda3/envs/etpe_env/lib/python3.12/site-packages/pmdarima/arima/auto.py:444: UserWarning: Input time-series is completely constant; returning a (0, 0, 0) ARMA.
  warnings.warn('Input time-series is completely constant; '


tunned arima order:  (2, 1, 0)
tunned arima order:  (0, 1, 0)
tunned arima order:  (1, 1, 0)
tunned arima order:  (1, 1, 0)
tunned arima order:  (0, 1, 1)
tunned arima order:  (2, 1, 1)
tunned arima order:  (0, 1, 0)
tunned arima order:  (0, 1, 1)
tunned arima order:  (0, 1, 0)
tunned arima order:  (0, 1, 0)
tunned arima order:  (1, 0, 0)
tunned arima order:  (0, 1, 0)
tunned arima order:  (0, 0, 1)
tunned arima order:  (0, 1, 0)
tunned arima order:  (0, 1, 1)
tunned arima order:  (2, 0, 0)
tunned arima order:  (0, 1, 0)
tunned arima order:  (1, 1, 0)
tunned arima order:  (1, 0, 0)
tunned arima order:  (1, 0, 0)
tunned arima order:  (0, 1, 0)
tunned arima order:  (3, 1, 0)
tunned arima order:  (1, 1, 0)
tunned arima order:  (1, 1, 0)
tunned arima order:  (1, 0, 0)
tunned arima order:  (0, 1, 0)
tunned arima order:  (0, 1, 0)
tunned arima order:  (1, 1, 0)
tunned arima order:  (2, 1, 1)
tunned arima order:  (0, 1, 1)
tunned arima order:  (0, 1, 0)


/Users/javier/miniconda3/envs/etpe_env/lib/python3.12/site-packages/statsmodels/tsa/statespace/sarimax.py:1908: RuntimeWarning: divide by zero encountered in reciprocal
  return np.roots(self.polynomial_reduced_ma)**-1


tunned arima order:  (2, 1, 2)
tunned arima order:  (0, 0, 1)
tunned arima order:  (0, 0, 1)
tunned arima order:  (1, 0, 0)
tunned arima order:  (1, 1, 2)
tunned arima order:  (2, 0, 0)
tunned arima order:  (2, 0, 1)
tunned arima order:  (1, 0, 0)
tunned arima order:  (0, 0, 2)
tunned arima order:  (0, 1, 1)
tunned arima order:  (2, 0, 1)
tunned arima order:  (1, 1, 1)


In [16]:
# Make sure ensemble_arima_df is sorted by iso_alpha_3 and year
ensemble_arima_df = ensemble_arima_df.sort_values(by=["iso_alpha_3", "year"]).reset_index(drop=True)
ensemble_arima_df.head()

,iso_alpha_3,future_id,year,pop_growth,renewable_energy_consumption_pct,forest_area_pct,gdp_growth_pct,gdp_per_capita_growth_pct,manufacturing_pct_of_gdp,log_pop_total,log_gdp_2021_ppp_intl_usd,log_gdp_per_capita_2021_ppp_intl_usd,log_imports_pct_of_gdp,log_industry_pct_of_gdp,log_exports_pct_of_gdp,log_gdp_2021_ppp_intl_usd_lag1,log_pop_total_lag1
0,ABW,id_ABW_1,2022,-0.337761,10.137118,2.333335,14.648925,5.147431,3.590876,11.578391,22.208832,10.533504,4.582513,2.671729,4.385407,22.027094,11.585263
1,ABW,id_ABW_2,2022,-0.567708,10.060231,2.333334,-21.025290,-15.826441,3.026786,11.581559,22.029903,10.448299,4.460326,2.667899,4.336606,22.230373,11.583867
2,ABW,id_ABW_3,2022,0.203099,9.263504,2.333333,15.050320,3.241112,4.263263,11.590321,22.132441,10.488039,4.405343,2.737095,4.425685,22.112499,11.589052
3,ABW,id_ABW_4,2022,-0.183731,8.400128,2.333334,2.579515,-16.528964,5.223261,11.574408,22.191765,10.416234,4.459498,2.699903,4.245553,22.040569,11.587132
4,ABW,id_ABW_5,2022,0.014547,7.832166,2.333333,15.905191,9.801830,4.551638,11.576392,22.066730,10.460997,4.381245,2.532409,4.444410,21.972521,11.579222


In [17]:
ensemble_arima_tuned_df = ensemble_arima_tuned_df.sort_values(by=["iso_alpha_3", "year"]).reset_index(drop=True)
ensemble_arima_tuned_df.head()

,iso_alpha_3,future_id,year,pop_growth,renewable_energy_consumption_pct,forest_area_pct,gdp_growth_pct,gdp_per_capita_growth_pct,manufacturing_pct_of_gdp,log_pop_total,log_gdp_2021_ppp_intl_usd,log_gdp_per_capita_2021_ppp_intl_usd,log_imports_pct_of_gdp,log_industry_pct_of_gdp,log_exports_pct_of_gdp,log_gdp_2021_ppp_intl_usd_lag1,log_pop_total_lag1
0,ABW,id_ABW_1,2022,0.011180,10.249731,1.235182,-19.904771,-5.647255,3.586348,11.586150,10.800272,10.596288,4.462780,2.778544,-1.589820,62.545847,11.584076
1,ABW,id_ABW_2,2022,0.236977,10.355814,-3.326228,-5.869793,-10.953483,3.305676,11.585583,-5.173376,10.382315,4.508036,2.733035,-1.828594,16.900272,11.579342
2,ABW,id_ABW_3,2022,0.176351,7.519600,3.028318,-6.194237,-36.827341,4.977344,11.577949,-51.649590,10.653256,4.411415,2.564951,4.399075,13.845269,11.580332
3,ABW,id_ABW_4,2022,-0.711291,9.807750,1.323381,-21.631173,-11.502659,4.199418,11.581591,-0.056646,10.533218,4.432784,2.712899,3.362859,38.191654,11.585214
4,ABW,id_ABW_5,2022,-0.068958,7.656978,-1.139114,-12.767746,-12.215128,3.912666,11.579341,22.978164,10.485457,4.458650,2.711908,-0.529746,0.661685,11.578339


In [18]:
# Save the ensemble dataframes to csv files
ensemble_arima_df.to_csv(os.path.join(ENSEMBLE_DIR_PATH, f"ensemble_arima_{n_scenarios}_df.csv"), index=False)
ensemble_arima_tuned_df.to_csv(os.path.join(ENSEMBLE_DIR_PATH, f"ensemble_arima_tuned_{n_scenarios}_df.csv"), index=False)  # Comment this line if you don't want to generate auto-tuned ARIMA models